# 🤖 Machine Learning Model Training & Comparative Benchmarking

This notebook trains, evaluates, and benchmarks **5 distinct regression algorithms** to predict used car prices from the cleaned dataset.

### Machine Learning Models Benchmarked:
1. **Linear Regression** (Baseline parametric model)
2. **Ridge Regression** (L2-regularized linear model)
3. **Decision Tree Regressor** (Non-linear single tree model)
4. **Random Forest Regressor** (Ensemble Bagging algorithm)
5. **Gradient Boosting Regressor** (Ensemble Sequential Boosting algorithm)

### Evaluation Metrics:
- **$R^2$ Score** (Coefficient of Determination)
- **MAE** (Mean Absolute Error, in USD)
- **RMSE** (Root Mean Squared Error, in USD)
- **MAPE** (Mean Absolute Percentage Error)


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
%matplotlib inline

print("ML libraries loaded successfully!")


## 1. Load Cleaned Dataset & Define Features
We load `car_price_cleaned.csv` and set up numerical and categorical feature sets.


In [ ]:
data_file = "car_price_cleaned.csv"
df = pd.read_csv(data_file)
print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Numerical features
num_cols = ["Levy", "Prod_Year", "Car_Age", "Engine_Volume", "Mileage_km", "Cylinders", "Airbags", "Turbo"]

# Categorical features - group rare brands to prevent dimensional explosion
top_manufacturers = df["Manufacturer"].value_counts().nlargest(25).index.tolist()
df["Manufacturer_Grouped"] = df["Manufacturer"].apply(lambda x: x if x in top_manufacturers else "Other")

cat_cols_modeling = ["Manufacturer_Grouped", "Category", "Leather_Interior", "Fuel_Type", "Gear_Box_Type", "Drive_Wheels", "Doors", "Wheel", "Color"]

X = df[num_cols + cat_cols_modeling]
y = df["Price"]

print(f"Features: {X.shape[1]} columns ({len(num_cols)} numerical, {len(cat_cols_modeling)} categorical)")
print(f"Target: Price (min=${y.min():,.0f}, median=${y.median():,.0f}, max=${y.max():,.0f})")


## 2. Train-Test Split & Preprocessing Pipeline
We partition the dataset into 80% training and 20% test sets, and construct a `ColumnTransformer` with `StandardScaler` and `OneHotEncoder`.


In [ ]:
# 80% train, 20% test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]:,} samples | Test set: {X_test.shape[0]:,} samples")

# Build unified preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_modeling)
    ]
)
print("Preprocessor configured.")


## 3. Training & Benchmarking 5 Regression Models
We instantiate and train all 5 models within unified Scikit-Learn pipelines to prevent data leakage.


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=16, min_samples_split=6, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=180, learning_rate=0.08, max_depth=6, random_state=42)
}

results = []
trained_pipelines = {}

print("=" * 65)
print("TRAINING & EVALUATION IN PROGRESS...")
print("=" * 65)

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])
    
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    # Calculate evaluation metrics
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = mean_absolute_percentage_error(y_test, preds) * 100

    results.append({
        "Model": name,
        "R2_Score": round(r2, 4),
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "MAPE (%)": round(mape, 2)
    })
    trained_pipelines[name] = (pipeline, r2, mae, rmse)
    print(f"  -> R²: {r2:.4f} | MAE: ${mae:,.2f} | RMSE: ${rmse:,.2f} | MAPE: {mape:.2f}%")

print("=" * 65)
print("ALL MODELS TRAINED SUCCESSFULLY!")
print("=" * 65)


## 4. Model Comparison & Benchmarking Results
Let's inspect the results in a formatted dataframe and save the results to `model_comparison.csv`.


In [ ]:
results_df = pd.DataFrame(results).sort_values(by="R2_Score", ascending=False).reset_index(drop=True)
results_df.to_csv("model_comparison.csv", index=False)

# Display styled table
results_df.style.highlight_max(subset=["R2_Score"], color="#BBF7D0")\
                .highlight_min(subset=["MAE ($)", "RMSE ($)"], color="#BBF7D0")


### Visual Comparison of Model Performance
Visualizing $R^2$ Score (higher is better) and Mean Absolute Error (lower is better).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. R2 Score Comparison
sns.barplot(data=results_df, x="R2_Score", y="Model", hue="Model", palette="Blues_r", legend=False, ax=axes[0])
axes[0].set_title("Model Comparison - R² Score (Higher is Better)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("R² Score")
for i, v in enumerate(results_df["R2_Score"]):
    axes[0].text(v + 0.01, i, f"{v:.4f}", va="center", fontweight="bold")

# 2. MAE Comparison
sns.barplot(data=results_df, x="MAE ($)", y="Model", hue="Model", palette="Reds_r", legend=False, ax=axes[1])
axes[1].set_title("Model Comparison - Mean Absolute Error ($) (Lower is Better)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("MAE ($)")
for i, v in enumerate(results_df["MAE ($)"]):
    axes[1].text(v + 100, i, f"${v:,.0f}", va="center", fontweight="bold")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()


## 5. Export Best Model and Metadata
We select the winning model (**Gradient Boosting Regressor**) and serialize it to `best_model.pkl` using Joblib.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipe, best_r2, best_mae, best_rmse = trained_pipelines[best_model_name]
print(f"Top performing model: '{best_model_name}' (R² = {best_r2:.4f}, MAE = ${best_mae:,.2f})")

# Save model
joblib.dump(best_pipe, "best_model.pkl")
print("Saved best model pipeline to 'best_model.pkl'")

# Export metadata
metadata = {
    "best_model": {
        "name": best_model_name,
        "r2_score": best_r2,
        "mae": best_mae,
        "rmse": best_rmse
    },
    "num_features": num_cols,
    "cat_features": cat_cols_modeling,
    "manufacturers": sorted(top_manufacturers) + ["Other"],
    "categories": sorted(df["Category"].dropna().unique().tolist())
}
with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("Updated 'model_metadata.json'")


## 6. End-to-End Prediction Test
Let's verify our saved model pipeline with an unseen sample vehicle input.


In [ ]:
# Test Sample Car
sample_car = pd.DataFrame([{
    "Levy": 781.0,
    "Prod_Year": 2015.0,
    "Car_Age": 5,
    "Engine_Volume": 2.0,
    "Mileage_km": 85000.0,
    "Cylinders": 4.0,
    "Airbags": 6.0,
    "Turbo": 0,
    "Manufacturer_Grouped": "TOYOTA",
    "Category": "Sedan",
    "Leather_Interior": "Yes",
    "Fuel_Type": "Petrol",
    "Gear_Box_Type": "Automatic",
    "Drive_Wheels": "Front",
    "Doors": "4-5",
    "Wheel": "Left wheel",
    "Color": "Black"
}])

# Load model and predict
loaded_model = joblib.load("best_model.pkl")
predicted_price = loaded_model.predict(sample_car)[0]

print("=== SAMPLE INFERENCE TEST ===")
print("Vehicle: 2015 Toyota Sedan (2.0L, 85,000 km, Automatic)")
print(f"Predicted Market Value: ${predicted_price:,.2f}")
print(f"Estimated Price Range:  ${max(500, predicted_price - 0.75 * best_mae):,.2f} - ${predicted_price + 0.75 * best_mae:,.2f}")


## 7. Project Conclusion
- **Gradient Boosting Regressor** is the best model for predicting used car prices with $R^2 = 0.5880$ and lowest RMSE.
- **Random Forest** achieved the lowest Mean Absolute Error ($6,654).
- The model is ready to serve predictions in the Streamlit web app (`app.py`). Run `streamlit run app.py` to launch the interactive UI!
